In [41]:
import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

randon_state = 42

print("libraries impoted successfully")

libraries impoted successfully


In [42]:
pfa_path = "../data/raw-data/pfa.json"
print("PFA Artice file:", pfa_path)

PFA Artice file: ../data/raw-data/pfa.json


In [43]:
if not os.path.exists(pfa_path):
    true_path = None
    
if pfa_path is None :
    raise FileNotFoundError(
        "Fake.csv and True.csv were not found."
    )

pfa_df = pd.read_json(pfa_path)
pfa_df['label'].unique()

<ArrowStringArray>
[        'false',   'mostly_true',     'half_true', 'pants_on_fire',
  'mostly_false',     'half_flip',          'true',     'full_flop',
       'no_flip']
Length: 9, dtype: str

In [44]:
# Step 1: Remove middle labels
pfa_df['label'] = pfa_df['label'].str.strip()
middle_labels = ['mostly_false','mostly_true', 'half_true', 'half_flip','full_flop','no_flip']
label_mapping = {
    'false': 0,
    'pants_on_fire': 0,
    'true': 1
}
rows_before = len(pfa_df)

pfa_df = pfa_df[~pfa_df['label'].isin(middle_labels)].copy()
pfa_df['label'] = pfa_df['label'].map(label_mapping)
rows_after = len(pfa_df)
rows_removed = rows_before - rows_after

print(f"\nRemoved: {rows_removed} rows")
print(f"Remaining: {rows_after} rows")

print("\nLabel distribution after removing middle labels:")
print(pfa_df['label'].unique())



Removed: 11348 rows
Remaining: 14381 rows

Label distribution after removing middle labels:
[0 1]


In [45]:
pfa_df['content'] = pfa_df['title']
pfa_df["content"] = pfa_df["content"].astype(str).str.strip()
pfa_df = pfa_df[pfa_df["content"].str.len() > 10].copy()
pfa_df = pfa_df[['content', 'label']].copy()

In [46]:
pfa_df = pfa_df.copy()
print(pfa_df['label'].value_counts().sort_index())
pfa_df.head(10)

label
0    11762
1     2619
Name: count, dtype: int64


,content,label
0,"Says Barack Obama tweeted, “Has anyone checked...",0
2,Former USAID Administrator Samantha Power’s ne...,0
4,The Kansas City Chiefs’ Super Bowl win has bee...,0
5,"Nationally, they predict prison populations on...",0
11,Says there are thousands of registered voters ...,0
12,Patent shows 5G towers are used for brainwashing.,0
13,We are experiencing the worst economic recover...,0
17,A campaign poster for New Zealand Prime Minist...,0
19,Why was there no news coverage of this raid on...,0
20,“CDC Director Robert Redfield says healthy peo...,0


In [47]:
pfa_df['content'][20][:]

'“CDC Director Robert Redfield says healthy people should NOT wear masks.”'

In [50]:
def clean_claim(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    
    # Optional: Remove standard PolitiFact prefix like "Says Donald Trump that..."
    text = re.sub(r"^says\s+[^,]+,\s*", " ", text, flags=re.IGNORECASE)
    
    text = text.lower()
    
    # Remove URLs and HTML
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    
    # Keep only alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text
pfa_df['clean_text'] = pfa_df['content'].apply(clean_claim)

pfa_df = pfa_df[pfa_df['clean_text'].str.split().str.len() >= 4].copy()
pfa_df = pfa_df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)

pfa_df["word_count"] = pfa_df["clean_text"].astype(str).str.split().str.len()
# 2. Check counts before filtering
rows_before = len(pfa_df)
# 3. Filter to keep only rows with word count >= 10
pfa_df = pfa_df[pfa_df["word_count"] >= 10].copy().reset_index(drop=True)
rows_after = len(pfa_df)
print(f"Rows before filter : {rows_before:,}")
print(f"Rows after filter  : {rows_after:,}")
print(f"Short rows removed : {rows_before - rows_after:,}")
# 4. View the new distribution


Rows before filter : 11,977
Rows after filter  : 11,977
Short rows removed : 0

Word count statistics:
count    11977.0
mean        17.6
std          6.7
min         10.0
25%         13.0
50%         16.0
75%         21.0
max         68.0
Name: word_count, dtype: float64
label
0    80.49
1    19.51
Name: proportion, dtype: float64


In [52]:
label_counts = pfa_df["label"].value_counts().sort_index()

print("class distribution:")
print(label_counts)

print("\nWord count statistics:")
print(pfa_df["word_count"].describe().round(1))
print(pfa_df["label"].value_counts(normalize=True).sort_index().mul(100).round(2))

class distribution:
label
0    9640
1    2337
Name: count, dtype: int64

Word count statistics:
count    11977.0
mean        17.6
std          6.7
min         10.0
25%         13.0
50%         16.0
75%         21.0
max         68.0
Name: word_count, dtype: float64
label
0    80.49
1    19.51
Name: proportion, dtype: float64


In [53]:
# Select final columns
clean_pfa = pfa_df[['clean_text', 'label']].copy()

print(f"\nFinal cleaned PolitiFact test set size: {len(clean_pfa):,}")
print("Class breakdown:")
print(clean_pfa['label'].value_counts(normalize=True).round(4) * 100)


Final cleaned PolitiFact test set size: 11,977
Class breakdown:
label
0    80.49
1    19.51
Name: proportion, dtype: float64


In [54]:
clean_pfa.head(10)

,clean_text,label
0,has anyone checked to make sure donald trump d...,0
1,former usaid administrator samantha power s ne...,0
2,the kansas city chiefs super bowl win has been...,0
3,nationally they predict prison populations on ...,0
4,we are experiencing the worst economic recover...,0
5,a campaign poster for new zealand prime minist...,0
6,why was there no news coverage of this raid on...,0
7,cdc director robert redfield says healthy peop...,0
8,ann kuster supports the government takeover of...,0
9,large machinery is required to move pounds of ...,0


In [55]:
output_dir = "../data/processed_data"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "clean_politifact_benchmark.csv")
clean_pfa.to_csv(output_path, index=False)
print(f"\n✅ PolitiFact benchmark saved to: {output_path}")


✅ PolitiFact benchmark saved to: ../data/processed_data/clean_politifact_benchmark.csv
